In [1]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.storagelevel import StorageLevel

# =========================================================
# PREPROCESS FINAL - FIXED WINDOW / COLD START / PARQUET
# =========================================================

# -----------------------------
# 0) Config
# -----------------------------
RAW_PATH = "/user/data/raw/*.parquet"
OUT_BASE = "/user/data/preprocess"

START_TS = "2025-01-01 00:00:00"
END_TS   = "2025-06-01 00:00:00"

BIN_MINUTES = 30
BINS_PER_DAY = 48
BINS_PER_WEEK = 7 * BINS_PER_DAY   # 336

ACTIVE_RATIO_THRESHOLD = 0.75
EWMA_ALPHA = 0.3
EWMA_LOOKBACK = 60                 # 60 bins = 30 hours

MIN_DURATION_MIN = 1.0
MAX_DURATION_MIN = 180.0           # 3 hours
MIN_TRIP_DISTANCE = 0.1
MAX_TRIP_DISTANCE = 50.0
MAX_PASSENGERS = 8

MIN_LOCATION_ID = 1
MAX_LOCATION_ID = 263

MONEY_COLS = ["fare_amount", "total_amount"]

# cold-start cutoff:
# để dùng model an toàn, cần đủ history cho tất cả feature lớn nhất
REQUIRED_HISTORY_BINS = max(5, BINS_PER_DAY, EWMA_LOOKBACK, BINS_PER_WEEK)  # = 336

spark = SparkSession.builder.appName("Preprocess").getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.mergeSchema", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "400")

# -----------------------------
# 1) Read raw parquet
# -----------------------------
raw = (
    spark.read
         .option("mergeSchema", "true")
         .parquet(RAW_PATH)
         .withColumn("_source_file", F.input_file_name())
)

print("=== RAW SCHEMA ===")
raw.printSchema()
print("Raw rows:", raw.count())
print("Raw files:", len(raw.inputFiles()))

required_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID"]
missing_required = [c for c in required_cols if c not in raw.columns]
if missing_required:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_required}")

# -----------------------------
# 2) Select + cast
# -----------------------------
df = raw.select(
    F.col("tpep_pickup_datetime").alias("pickup_dt"),
    F.col("tpep_dropoff_datetime").alias("dropoff_dt"),
    F.col("PULocationID").cast("int").alias("PULocationID"),
    F.col("DOLocationID").cast("int").alias("DOLocationID") if "DOLocationID" in raw.columns else F.lit(None).cast("int").alias("DOLocationID"),
    F.col("passenger_count").cast("int").alias("passenger_count") if "passenger_count" in raw.columns else F.lit(None).cast("int").alias("passenger_count"),
    F.col("trip_distance").cast("double").alias("trip_distance") if "trip_distance" in raw.columns else F.lit(None).cast("double").alias("trip_distance"),
    F.col("fare_amount").cast("double").alias("fare_amount") if "fare_amount" in raw.columns else F.lit(None).cast("double").alias("fare_amount"),
    F.col("total_amount").cast("double").alias("total_amount") if "total_amount" in raw.columns else F.lit(None).cast("double").alias("total_amount"),
    F.col("_source_file")
)

# -----------------------------
# 3) Time filter
# -----------------------------
df = (
    df.filter(F.col("pickup_dt").isNotNull() & F.col("dropoff_dt").isNotNull())
      .filter(
          (F.col("pickup_dt") >= F.lit(START_TS).cast("timestamp")) &
          (F.col("pickup_dt") <  F.lit(END_TS).cast("timestamp"))
      )
)

# -----------------------------
# 4) Cleaning
# -----------------------------
df = df.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("dropoff_dt") - F.unix_timestamp("pickup_dt")) / 60.0
)

df = (
    df.filter(F.col("PULocationID").between(MIN_LOCATION_ID, MAX_LOCATION_ID))
      .filter(F.col("trip_duration_min").between(MIN_DURATION_MIN, MAX_DURATION_MIN))
      .filter(F.col("trip_distance").isNotNull())
      .filter(F.col("trip_distance").between(MIN_TRIP_DISTANCE, MAX_TRIP_DISTANCE))
)

if "passenger_count" in df.columns:
    df = df.filter(
        F.col("passenger_count").isNull() |
        ((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= MAX_PASSENGERS))
    )

for c in MONEY_COLS:
    if c in df.columns:
        df = df.filter(F.col(c).isNull() | (F.col(c) >= 0))

print("Rows after cleaning:", df.count())

# -----------------------------
# 5) 30-minute pickup demand aggregation
# -----------------------------
df = df.withColumn(
    "pickup_bin_30m",
    F.to_timestamp(
        F.from_unixtime(
            F.floor(F.unix_timestamp("pickup_dt") / (BIN_MINUTES * 60)) * (BIN_MINUTES * 60)
        )
    )
)

agg = (
    df.groupBy("pickup_bin_30m", "PULocationID")
      .agg(F.count("*").alias("pickup_demand"))
)

# -----------------------------
# 6) Dense panel = all time bins x active locations candidate
# -----------------------------
full_bins = spark.sql(f"""
SELECT explode(
    sequence(
        to_timestamp('{START_TS}'),
        to_timestamp('{END_TS}') - interval 30 minutes,
        interval 30 minutes
    )
) AS pickup_bin_30m
""")

all_locations = agg.select("PULocationID").distinct()
dense_grid = full_bins.crossJoin(all_locations)

dense_panel = (
    dense_grid.join(agg, ["pickup_bin_30m", "PULocationID"], "left")
              .fillna({"pickup_demand": 0})
              .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Dense panel rows before active-location filter:", dense_panel.count())

# -----------------------------
# 7) Time-based split boundaries
# -----------------------------
full_bounds = full_bins.select(
    F.min("pickup_bin_30m").alias("min_ts"),
    F.max("pickup_bin_30m").alias("max_ts")
).first()

min_ts = full_bounds["min_ts"]
max_ts = full_bounds["max_ts"]

seconds_total = int(max_ts.timestamp() - min_ts.timestamp())
train_cut_sec = int(seconds_total * 0.70)
val_cut_sec   = int(seconds_total * 0.90)

train_cut_expr = f"timestamp'{min_ts.strftime('%Y-%m-%d %H:%M:%S')}' + interval {train_cut_sec} seconds"
val_cut_expr   = f"timestamp'{min_ts.strftime('%Y-%m-%d %H:%M:%S')}' + interval {val_cut_sec} seconds"

print("Train cut expr:", train_cut_expr)
print("Val cut expr:", val_cut_expr)

# -----------------------------
# 8) Active locations selected from TRAIN ONLY
# -----------------------------
train_dense_panel = dense_panel.filter(F.col("pickup_bin_30m") < F.expr(train_cut_expr))

active_locations = (
    train_dense_panel.groupBy("PULocationID")
                     .agg(
                         F.sum(F.when(F.col("pickup_demand") > 0, 1).otherwise(0)).alias("active_bins"),
                         F.count("*").alias("all_bins")
                     )
                     .withColumn("active_ratio", F.col("active_bins") / F.col("all_bins"))
                     .filter(F.col("active_ratio") >= ACTIVE_RATIO_THRESHOLD)
                     .select("PULocationID")
                     .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Selected active locations (train-only):", active_locations.count())

dense_panel = (
    dense_panel.join(active_locations, "PULocationID", "inner")
               .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Dense panel rows after active-location filter:", dense_panel.count())

# Có thể unpersist bản cũ để tiết kiệm RAM
train_dense_panel.unpersist()
active_locations.unpersist()

# -----------------------------
# 9) Calendar/time features
# IMPORTANT:
# Không dùng Window.orderBy(...) global để tạo time_idx nữa
# => tránh WARN "No Partition Defined for Window operation"
# -----------------------------
panel = (
    dense_panel
    .withColumn("date", F.to_date("pickup_bin_30m"))
    .withColumn("hour", F.hour("pickup_bin_30m").cast("int"))
    .withColumn("minute", F.minute("pickup_bin_30m").cast("int"))
    .withColumn("slot_30m", (F.col("hour") * 2 + F.floor(F.col("minute") / 30)).cast("int"))
    .withColumn("day_of_week", ((F.dayofweek("pickup_bin_30m") + 5) % 7).cast("int"))  # Monday=0
    .withColumn("week_of_year", F.weekofyear("pickup_bin_30m").cast("int"))
    .withColumn("month", F.month("pickup_bin_30m").cast("int"))
    .withColumn("day_of_month", F.dayofmonth("pickup_bin_30m").cast("int"))
    .withColumn("is_weekday", F.when(F.col("day_of_week").between(0, 4), 1).otherwise(0).cast("int"))
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin([5, 6]), 1).otherwise(0).cast("int"))
    .withColumn(
        "time_idx",
        (
            (F.unix_timestamp("pickup_bin_30m") - F.unix_timestamp(F.lit(START_TS).cast("timestamp")))
            / (BIN_MINUTES * 60)
        ).cast("int")
    )
)

# -----------------------------
# 10) Partitioned windows (đã sửa triệt để)
# -----------------------------
w_loc = Window.partitionBy("PULocationID").orderBy("time_idx")
w_24h = Window.partitionBy("PULocationID").orderBy("time_idx").rowsBetween(-48, -1)

feat = panel

# lags
for i in range(1, 6):
    feat = feat.withColumn(f"demand_t_{i}", F.lag("pickup_demand", i).over(w_loc))

# rolling
feat = (
    feat.withColumn("rolling_obs_24h", F.count("pickup_demand").over(w_24h))
        .withColumn("rolling_max_24h", F.max("pickup_demand").over(w_24h))
        .withColumn("rolling_min_24h", F.min("pickup_demand").over(w_24h))
        .withColumn("rolling_mean_24h", F.avg("pickup_demand").over(w_24h))
        .withColumn("rolling_std_24h", F.stddev_samp("pickup_demand").over(w_24h))
        .withColumn("rolling_skew_24h", F.skewness("pickup_demand").over(w_24h))
)

# Historical Average = exactly 1 week back
feat = feat.withColumn("ha_output", F.lag("pickup_demand", BINS_PER_WEEK).over(w_loc))

# -----------------------------
# 11) Native Spark EWMA
# -----------------------------
ewma_expr = F.lit(0.0)
for i in range(1, EWMA_LOOKBACK + 1):
    weight = EWMA_ALPHA * ((1.0 - EWMA_ALPHA) ** (i - 1))
    ewma_expr = ewma_expr + (
        F.coalesce(F.lag("pickup_demand", i).over(w_loc), F.lit(0.0)).cast("double") * F.lit(weight)
    )

feat = feat.withColumn("ewma_output", ewma_expr.cast("double"))

# -----------------------------
# 12) Fill nulls for lag/rolling/HA
# -----------------------------
fill_map = {
    "demand_t_1": 0.0,
    "demand_t_2": 0.0,
    "demand_t_3": 0.0,
    "demand_t_4": 0.0,
    "demand_t_5": 0.0,
    "rolling_obs_24h": 0,
    "rolling_max_24h": 0.0,
    "rolling_min_24h": 0.0,
    "rolling_mean_24h": 0.0,
    "rolling_std_24h": 0.0,
    "rolling_skew_24h": 0.0,
    "ha_output": 0.0,
    "ewma_output": 0.0,
}
feat = feat.fillna(fill_map)

# -----------------------------
# 13) Split labels
# -----------------------------
feat = feat.withColumn(
    "dataset_split",
    F.when(F.col("pickup_bin_30m") < F.expr(train_cut_expr), "train")
     .when(F.col("pickup_bin_30m") < F.expr(val_cut_expr), "validation")
     .otherwise("test")
)

# -----------------------------
# 14) Cold-start flags
# - giữ lại full panel để audit
# - tạo thêm model_ready để train an toàn
# -----------------------------
feat = (
    feat.withColumn("history_bins_available", F.col("time_idx"))
        .withColumn("enough_history_lag5", (F.col("time_idx") >= 5).cast("int"))
        .withColumn("enough_history_24h", (F.col("time_idx") >= BINS_PER_DAY).cast("int"))
        .withColumn("enough_history_ewma", (F.col("time_idx") >= EWMA_LOOKBACK).cast("int"))
        .withColumn("enough_history_1w", (F.col("time_idx") >= BINS_PER_WEEK).cast("int"))
        .withColumn("is_cold_start", (F.col("time_idx") < REQUIRED_HISTORY_BINS).cast("int"))
        .withColumn("model_ready", (F.col("time_idx") >= REQUIRED_HISTORY_BINS).cast("int"))
)

# -----------------------------
# 15) Final panel
# -----------------------------
final_panel = feat.select(
    "pickup_bin_30m",
    "date",
    "time_idx",
    "dataset_split",
    "PULocationID",

    # target
    "pickup_demand",

    # time features
    "hour",
    "minute",
    "slot_30m",
    "day_of_week",
    "week_of_year",
    "month",
    "day_of_month",
    "is_weekday",
    "is_weekend",

    # lag features
    "demand_t_1",
    "demand_t_2",
    "demand_t_3",
    "demand_t_4",
    "demand_t_5",

    # rolling features
    "rolling_obs_24h",
    "rolling_max_24h",
    "rolling_min_24h",
    "rolling_mean_24h",
    "rolling_std_24h",
    "rolling_skew_24h",

    # baseline features
    "ha_output",
    "ewma_output",

    # cold start / readiness flags
    "history_bins_available",
    "enough_history_lag5",
    "enough_history_24h",
    "enough_history_ewma",
    "enough_history_1w",
    "is_cold_start",
    "model_ready"
).persist(StorageLevel.MEMORY_AND_DISK)

model_panel = final_panel.filter(F.col("model_ready") == 1).persist(StorageLevel.MEMORY_AND_DISK)

# -----------------------------
# 16) Sanity checks
# -----------------------------
print("\n=== FINAL PANEL SCHEMA ===")
final_panel.printSchema()

print("\nFinal rows (full):", final_panel.count())
print("Final rows (model_ready):", model_panel.count())
print("Final active locations:", final_panel.select("PULocationID").distinct().count())

print("\nRows by split (full):")
final_panel.groupBy("dataset_split").count().orderBy("dataset_split").show(truncate=False)

print("\nRows by split (model_ready only):")
model_panel.groupBy("dataset_split").count().orderBy("dataset_split").show(truncate=False)

print("\nCold-start report:")
final_panel.groupBy("dataset_split", "is_cold_start").count().orderBy("dataset_split", "is_cold_start").show(truncate=False)

print("\nPreview full:")
final_panel.orderBy("pickup_bin_30m", "PULocationID").show(10, truncate=False)

print("\nPreview model_ready:")
model_panel.orderBy("pickup_bin_30m", "PULocationID").show(10, truncate=False)

print("\nNull report (full):")
final_panel.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in final_panel.columns
]).show(truncate=False)

print("\nPickup demand summary (full):")
final_panel.select(
    F.min("pickup_demand").alias("min_demand"),
    F.max("pickup_demand").alias("max_demand"),
    F.avg("pickup_demand").alias("avg_demand")
).show(truncate=False)

print("\nRows per month (full):")
final_panel.groupBy(
    F.date_trunc("month", "pickup_bin_30m").alias("month_bucket")
).count().orderBy("month_bucket").show(truncate=False)

# -----------------------------
# 17) Write parquet
# - full_panel: giữ toàn bộ để audit / EDA
# - model_ready: dùng train model, đã loại cold-start
# -----------------------------
(
    final_panel.write
               .mode("overwrite")
               .partitionBy("dataset_split")
               .parquet(f"{OUT_BASE}/full_panel")
)

(
    model_panel.write
               .mode("overwrite")
               .partitionBy("dataset_split")
               .parquet(f"{OUT_BASE}/model_ready")
)

print(f"\n[DONE] Saved full panel  : {OUT_BASE}/full_panel")
print(f"[DONE] Saved model panel : {OUT_BASE}/model_ready")

# -----------------------------
# 18) Cleanup cache
# -----------------------------
dense_panel.unpersist()
final_panel.unpersist()
model_panel.unpersist()

# -----------------------------
# 19) Write manifest
# -----------------------------
manifest = {
    "raw_path": RAW_PATH,
    "output_full_panel": f"{OUT_BASE}/full_panel",
    "output_model_ready": f"{OUT_BASE}/model_ready",
    "start_ts": START_TS,
    "end_ts": END_TS,
    "bin_minutes": BIN_MINUTES,
    "bins_per_day": BINS_PER_DAY,
    "bins_per_week": BINS_PER_WEEK,
    "required_history_bins": REQUIRED_HISTORY_BINS,
    "active_ratio_threshold": ACTIVE_RATIO_THRESHOLD
}

manifest_path = f"/workspace/code/kshape/preprocess/manifest.json"
try:
    import json
    import os
    os.makedirs(os.path.dirname(manifest_path), exist_ok=True)
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=4)
except Exception as e:
    print(f"[WARN] Không ghi được manifest local json: {e}")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c7133a5e-49a1-40e9-a913-5f374b7bc1ba;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 103ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

=== RAW SCHEMA ===
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- _source_file: string (nullable = false)



Raw rows: 24083384
Raw files: 6


Rows after cleaning: 17874060


...

26/03/31 03:49:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



=== FINAL PANEL SCHEMA ===
root
 |-- pickup_bin_30m: timestamp (nullable = false)
 |-- date: date (nullable = false)
 |-- time_idx: integer (nullable = true)
 |-- dataset_split: string (nullable = false)
 |-- PULocationID: integer (nullable = true)
 |-- pickup_demand: long (nullable = false)
 |-- hour: integer (nullable = false)
 |-- minute: integer (nullable = false)
 |-- slot_30m: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- week_of_year: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- day_of_month: integer (nullable = false)
 |-- is_weekday: integer (nullable = false)
 |-- is_weekend: integer (nullable = false)
 |-- demand_t_1: long (nullable = true)
 |-- demand_t_2: long (nullable = true)
 |-- demand_t_3: long (nullable = true)
 |-- demand_t_4: long (nullable = true)
 |-- demand_t_5: long (nullable = true)
 |-- rolling_obs_24h: long (nullable = false)
 |-- rolling_max_24h: long (nullable = true)
 |-- rolling_min_24h: long (nulla


Final rows (full): 478236


Final rows (model_ready): 456060
Final active locations: 66

Rows by split (full):
+-------------+------+
|dataset_split|count |
+-------------+------+
|test         |47322 |
|train        |335280|
|validation   |95634 |
+-------------+------+


Rows by split (model_ready only):
+-------------+------+
|dataset_split|count |
+-------------+------+
|test         |47322 |
|train        |313104|
|validation   |95634 |
+-------------+------+


Cold-start report:
+-------------+-------------+------+
|dataset_split|is_cold_start|count |
+-------------+-------------+------+
|test         |0            |47322 |
|train        |0            |313104|
|train        |1            |22176 |
|validation   |0            |95634 |
+-------------+-------------+------+


Preview full:
+-------------------+----------+--------+-------------+------------+-------------+----+------+--------+-----------+------------+-----+------------+----------+----------+----------+----------+----------+----------+----------+--


[DONE] Saved full panel  : /user/data/preprocess/full_panel
[DONE] Saved model panel : /user/data/preprocess/model_ready
